We have so far only used the re.match function which tries to find a match at the beginning of a string.

The function re.search allows to match any substring of a string.

Example: ```re.search(r'\bback\b', s)``` will match strings "back", "a back, is a body part", "get back".

But it will not match the strings "backspace" or "comeback".

The function ```re.search``` finds only the first occurence.

We can use the ```re.findall``` function to find all occurences.

Let’s say we want to find all present participle words in a string s. The present participle words have ending 'ing'. The function call would look like this:

```re.findall(r'\w+ing\b', s)```.

Let’s try running this:

In [2]:
import re

In [3]:
s="Doing things, going home, staying awake, sleeping later"
re.findall(r'\w+ing\b',s)

['Doing', 'going', 'staying', 'sleeping']

Let’s say we want to pick up all the integers from a string. We can try that with the following function call: ```re.findall(r'[+-]?\d+', s)```. An example run:

In [5]:
s= "23 + -24 = -1"
re.findall(r'[+-]?\d+',s)

['23', '-24', '-1']

Suppose we are given a string of if/then sentences, and we would like to extract the conditions from these sentences. Let’s try the following function call:

In [3]:
s = ("If I’m not in a hurry, then I should stay. " +
    "On the other hand, if I leave, then I can sleep.")
re.findall(r'[Ii]f (.*), then', s)

['I’m not in a hurry, then I should stay. On the other hand, if I leave']

## Breaking Down `r'[Ii]f (.*), then'`

Every piece here is something you've already learned — let's snap the LEGO bricks together.

---

### Piece by Piece

| Piece | You know this as... | Meaning here |
|---|---|---|
| `[Ii]` | bracket set | one character: `I` **or** `i` |
| `f` | literal | the letter f |
| ` ` | literal | a space |
| `(.*)` | group + dot + star | **capture** any characters, any amount |
| `,` | literal | a comma |
| ` then` | literal | space + the word "then" |

So the pattern reads:

> *"Find `If` or `if`, a space, then **capture everything** up to a comma followed by ` then`."*

It extracts the **condition part of an if-then sentence**.

---

### Zooming Into `(.*)` — The Interesting Bit

Three familiar symbols stacked:

```
(   .   *   )
│   │   │   │
│   │   │   └─ close group
│   │   └─ zero or more of the previous thing
│   └─ any single character
└─ open group: CAPTURE what matches inside
```

- `.` = any character (your wildcard)
- `.*` = any character repeated **zero or more** times → *"anything, of any length, even nothing"*
- `(...)` = remember it — and here's the `findall` twist: **when the pattern contains a group, `findall` returns only the captured parts**, not the whole matches!

---

### Seeing It Run

```python
import re

s = "If it rains, then we stay home. if you study, then you pass."

re.findall(r'[Ii]f (.*?), then', s)      # (see note on ? below)
# → ['it rains', 'you study']
```

The full matches were `"If it rains, then"` and `"if you study, then"` — but `findall` handed back just the group contents: the **conditions**. That's the whole point of placing the parentheses exactly there — the fixed scaffolding (`If ... , then`) locates the sentence, the group extracts the variable middle.

---

### One Honest Warning — `.*` Is Greedy!

`*` matches **as much as possible**. With multiple `, then`s in one string, `.*` overshoots:

```python
s = "If A, then B. If C, then D."

re.findall(r'[Ii]f (.*), then', s)
# → ['A, then B. If C']            ← 😱 gobbled up to the LAST ', then'!

re.findall(r'[Ii]f (.*?), then', s)
# → ['A', 'C']                     ← *? = lazy: stop at the FIRST ', then' ✓
```

`.*?` (star followed by `?`) is the **non-greedy** version — *"any amount, but as little as possible."* A different job for `?` than the "optional" meaning you learned; after a repetition symbol it means *lazy mode*. For extract-between-markers patterns like this one, the lazy `.*?` is almost always what you actually want.

---

### The One-Line Summary

> `[Ii]f (.*), then` = *"case-tolerant `if`, then capture the condition sitting before `, then`"* — scaffolding locates, group extracts, and `findall` returns just the captured conditions. Add the `?` for the lazy version whenever the text might contain several `, then`s. 🎯

But I wanted a result: ["I'm not in a hurry", 'I leave']. That is, the condition from both sentences. How can this be fixed?

The problem is that the pattern .```*``` tries to match as many characters as possible. This is called greedy matching. One way of solving this problem is to notice that the two sentences are separated by a full-stop (.). So, instead of matching all the characters, we need to match everything but the dot character. This can be achieved by using the complement character class: ```[^.]```. The hat character (^) in the beginning of a character class means the complement character class

After the modification the function call looks like this

In [4]:
re.findall(r'[Ii]f ([^.]*), then',s)

['I’m not in a hurry', 'I leave']

## Breaking Down `r'[Ii]f ([^.]*), then'`

This is the **same pattern as before with one brick swapped**: `.*` became `[^.]*`. That tiny change is the whole story — and it's a clever fix for the greediness problem we just discussed!

---

### The One Changed Piece

```
Before:   ( . * )      capture: ANY character, any amount
Now:      ( [^.] * )   capture: any character EXCEPT '.', any amount
```

Zooming into `[^.]`:

| Symbol | You know this as... | Meaning here |
|---|---|---|
| `[^...]` | complement set (hat inside brackets) | any character NOT listed |
| `.` | ...wait, the wildcard? | **No! Inside brackets, `.` is just a literal dot** |

**Key subtlety:** inside `[...]`, most special characters lose their powers. So `[^.]` doesn't mean "not-anything" (which would be nonsense) — it means **"any character except a period."**

So the full pattern reads:

> *"`If` or `if`, space, then capture any run of characters **that contains no period**, up to a comma followed by ` then`."*

---

### Why Forbid the Period? — Sentence Containment!

Recall the greedy disaster from last time:

```python
import re
s = "If A, then B. If C, then D."

re.findall(r'[Ii]f (.*), then', s)
# → ['A, then B. If C']          ← .* rampaged across the sentence border!
```

The capture crossed the `.` between sentences and swallowed a whole extra sentence. Now watch the new version:

```python
re.findall(r'[Ii]f ([^.]*), then', s)
# → ['A', 'C']                    ✓ each condition from its own sentence!
```

**How it works:** when the capture reaches the period after `B`, `[^.]` **refuses to match it** — the run is forced to stop *within* the sentence. The period acts as a wall 🧱 the capture cannot pass. Backtracking then settles on the `, then` inside the same sentence.

---

### Three Solutions, One Problem — Compare

You've now seen three variants of this pattern:

| Pattern | Strategy | On `"If A, then B. If C, then D."` |
|---|---|---|
| `(.*)`  | greedy — grab maximally | `['A, then B. If C']` ✗ |
| `(.*?)` | lazy — grab minimally | `['A', 'C']` ✓ |
| `([^.]*)` | **fenced** — grab freely, but never cross a `.` | `['A', 'C']` ✓ |

The last two agree here, but they encode different *ideas*:

- `.*?` says: *stop at the earliest possible `, then`*
- `[^.]*` says: *stay inside one sentence*

They can differ! If a condition legitimately contained a period (say, an abbreviation like `"e.g."` or a number `"3.5"`), `[^.]*` would refuse it while `.*?` would allow it:

```python
s2 = "If x > 3.5, then stop."
re.findall(r'[Ii]f (.*?), then', s2)     # → ['x > 3.5']   ✓
re.findall(r'[Ii]f ([^.]*), then', s2)   # → []            ✗ the 3.5 dot blocked it!
```

Neither is universally "better" — they encode different assumptions about the text. Choosing between them **is** the design decision.

---

### The Takeaway

> `[^.]*` = *"any amount of period-free text"* — a **fenced capture** that can't leak across sentence boundaries. The complement-set trick `[^X]*` is the classic idiom for *"grab everything up to, but never through, character X"* — you'll see it constantly as `[^"]*` (inside quotes), `[^>]*` (inside HTML tags), `[^,]*` (one CSV field).
>
> And a bonus rule learned: **inside brackets, the dot is just a dot** — special characters go off-duty inside `[...]`. 🎯

## What That Paragraph Means — Simply

It's describing a **division of labor** inside the pattern. Let me make it concrete.

---

### The Pattern Has Two Kinds of Parts

```
[Ii]f (.*?), then
─────┬───── ──┬──
     │        │
scaffolding   │  scaffolding
        ┌─────┘
     the group
```

**Scaffolding** (outside the parentheses): `If ... , then` — the **fixed** words. Their job is to **locate** the right spot in the text. Like map coordinates: *"go to where a sentence has this shape."*

**Group** (inside the parentheses): the **variable** middle part — different in every sentence. Its job is to be **extracted**.

---

### The Search vs. The Catch

Given:

```python
s = "If it rains, then we stay home."
```

The **whole pattern** matches this much text:

```
If it rains, then
└───────┬────────┘
   full match — everything the pattern touched
```

But the parentheses were only around the middle:

```
If  (it rains)  , then
     └───┬───┘
      the captured part
```

And here's the `findall` rule the paragraph is pointing at:

> **When the pattern contains a group, `findall` returns only what the group captured** — not the full match.

```python
re.findall(r'[Ii]f (.*?), then', s)
# → ['it rains']         ← just the group — 'If' and ', then' discarded!
```

---

### A Fishing Analogy 🎣

> - The **scaffolding** `If ... , then` is your **fishing spot** — it tells you *where* the fish are.
> - The **group** `(...)` is the **net** — it decides *what you keep*.
> - `findall` hands you **only what's in the net**.
>
> You needed the whole rig to find the right place — but you never wanted to take the riverbank home. `If` and `, then` were only there to *find* the sentence; the condition between them is the *prize*.

---

### Why This Is "The Whole Point" of the Parentheses' Placement

You could have parenthesized differently — placement chooses the prize:

```python
s = "If it rains, then we stay home."

re.findall(r'[Ii]f (.*?), then', s)       # → ['it rains']        capture the condition
re.findall(r'[Ii]f .*?, then (.*?)\.', s) # → ['we stay home']    capture the consequence
re.findall(r'[Ii]f .*?, then', s)         # → ['If it rains, then']  no group → full match
```

Same locating logic each time — but *where you put the net* determines what `findall` returns. The parentheses are how you say: **"out of everything the pattern touches, THIS is the part I actually want."**

---

### The One-Sentence Version

> The fixed words **find** the sentence; the parentheses **pick** which slice of it you get back. Pattern = address, group = package. `findall` delivers only the package. 🎯

Another way of solving this problem is to use a non-greedy matching. The repetition specifiers ```+, *, ?, and {m,n}``` have corresponding non-greedy versions: ```+?, *?, ??, and {m,n}?```. These expressions use as few characters as possible to make the whole pattern match some substring

## What This Passage Says — Simply

Good news: you've **already met both solutions** — this passage is your textbook formally presenting what we discovered together in the last few questions! Let me connect it up and add the one new bit.

---

### The Problem Being Solved (Recap)

Repetition symbols are **greedy** by default — they grab as much as possible:

```python
s = "If A, then B. If C, then D."

re.findall(r'[Ii]f (.*), then', s)
# → ['A, then B. If C']            ← .* overshot to the LAST ', then'
```

The textbook offers **two fixes**, both of which you've seen:

---

### Fix 1 — The Fence: `[^.]*`

```python
re.findall(r'[Ii]f ([^.]*), then', s)
# → ['A', 'C']
```

*"Any characters, but never a period"* — the capture physically can't cross a sentence boundary. This was your previous question.

---

### Fix 2 — Non-Greedy Matching: `.*?`

```python
re.findall(r'[Ii]f (.*?), then', s)
# → ['A', 'C']
```

Adding `?` **after a repetition symbol** flips it into lazy mode: *"as few characters as possible while still letting the whole pattern succeed."*

The mindset difference:

```
Greedy  .*   :  grab EVERYTHING → then back off until ', then' fits
Lazy    .*?  :  grab NOTHING → then reluctantly eat one char at a time
                until ', then' fits
```

Both stop when the full pattern matches — they just approach from opposite ends. Greedy finds the **last** `, then`; lazy finds the **first**.

---

### The One NEW Thing — It's a Complete Family

What the passage adds beyond our discussions: **every** repetition symbol has a lazy twin, not just `*`:

| Greedy | Lazy | Meaning (lazy version) |
|---|---|---|
| `*` | `*?` | zero or more — as few as possible |
| `+` | `+?` | one or more — as few as possible |
| `?` | `??` | optional — prefer **zero** |
| `{m,n}` | `{m,n}?` | m to n — prefer **m** |

Quick demos of the less obvious ones:

```python
re.search(r'ab+?', 'abbb').group()      # → 'ab'      one b is enough — stop!
re.search(r'ab+',  'abbb').group()      # → 'abbb'    greedy takes all

re.search(r'a{2,4}?', 'aaaa').group()   # → 'aa'      minimum of the range
```

And yes — `??` looks odd, but it parses as: first `?` = "optional", second `?` = "lazily so" (prefer skipping it). Same rule as everywhere: **a trailing `?` on a repetition means lazy**.

This also resolves the double life of `?` you noticed earlier:

```
c?        after a normal token   → OPTIONAL (0 or 1)
*? +? ??  after a repetition     → LAZY MODE switch
```

---

### Which Fix Should You Use?

The textbook presents them as alternatives — recall from our fence discussion they're not always interchangeable:

```python
s2 = "If x > 3.5, then stop."
re.findall(r'[Ii]f (.*?), then', s2)     # → ['x > 3.5']  ✓ dot allowed
re.findall(r'[Ii]f ([^.]*), then', s2)   # → []           ✗ dot forbidden!
```

> - `[^X]*` — **fence**: *"stay within a region bounded by X"* (sentences, quotes, tags)
> - `*?` — **lazy**: *"stop at the earliest possible end-marker"*
>
> Same output on simple text, different philosophy — pick the one whose *assumption* matches your data. 🎯

---

### One-Sentence Summary

> Greedy repetition (`* + ? {m,n}`) takes the maximum; append `?` to any of them (`*? +? ?? {m,n}?`) and it takes the **minimum** — the standard cure for a capture that overshoots to the last delimiter instead of stopping at the first.

In [6]:
re.findall(r'[Ii]f (.*?), then', s)

['I’m not in a hurry', 'I leave']